<a href="https://colab.research.google.com/github/varba187/RAGs-to-Riches/blob/main/bart_baseline_train_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install transformers datasets sentencepiece accelerate evaluate

In [ ]:
import re
import torch
import transformers
import pandas as pd

from datasets import load_dataset
from transformers import (
    BartTokenizer,
    BartForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(F"Device set to {device}")

torch.manual_seed(0)

# Load NQ

In [ ]:
dataset = load_dataset("sentence-transformers/natural-questions", split="train")
print(dataset)
print(dataset.column_names)
print(dataset[0])

# First, let's train on a small dataset

In [ ]:
small_dataset = dataset.select(range(5000))
split_dataset = small_dataset.train_test_split(test_size=0.2)

train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

In [ ]:
model_name = "facebook/bart-large"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)
model = model.to(device)

# Data preprocessing

In [ ]:
max_input_length = 128
max_output_length = 32

def get_inputs(dataset_entry):
  question = dataset_entry["query"]
  answer = dataset_entry["answer"]

  if (isinstance(answer, list)):
    answer = answer[0]

  model_inputs = tokenizer(question, max_length = max_input_length, truncation = True)
  labels = tokenizer(answer, max_length = max_output_length, truncation = True)
  model_inputs["labels"] = labels["input_ids"]

  return model_inputs

In [ ]:
tokenized_train = train_dataset.map(get_inputs, remove_columns=train_dataset.column_names)
tokenized_test = test_dataset.map(get_inputs, remove_columns=test_dataset.column_names)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Here we define some important arguments for training

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    num_train_epochs=1,
    logging_strategy="steps",
    logging_steps=100,
    predict_with_generate=True,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    save_total_limit=1,
    report_to="none"
)

# Normalization

In [ ]:
def normalize_text(text):
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

# Exact Match Metrics

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    decoded_pred = tokenizer.batch_decode(predictions, skip_special_tokens = True)
    labels = [[token if token != -100 else tokenizer.pad_token_id for token in label] for label in labels]
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens = True)

    em = [int(normalize_text(pred) == normalize_text(label)) for pred, label in zip(decoded_pred, decoded_labels)]

    return {"em": 100 * sum(em) / len(em)}

# Training

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
metrics = trainer.evaluate()
print(metrics)

# Predictions generation

In [ ]:
predictions = trainer.predict(tokenized_test)
print(predictions.predictions.shape, predictions.label_ids.shape)

In [ ]:
preds, labels = predictions.predictions, predictions.label_ids
decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

labels = [[token if token != -100 else tokenizer.pad_token_id for token in label] for label in labels]
decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

results_df = pd.DataFrame({"question": test_dataset["query"], "answer": decoded_labels, "prediction": decoded_preds})
results_df["em"] = [int(normalize_text(prediction) == normalize_text(result)) for prediction, result in zip(results_df["prediction"], results_df["answer"])]

In [ ]:
results_df.to_csv('bart_baseline_train_eval_results.csv', index=False)
print("Results saved to bart_baseline_train_eval_results.csv")
results_df.head()